# 06 — Results Verification and Reporting

Checks saved evaluation artifacts, recalculates test metrics from saved checkpoints, and prepares project result summaries.

**Environment:** Google Colab with Google Drive mounted. Check paths and prerequisite artifacts before running.

This notebook contains verification and project-inspection cells from the original workflow. Some cells are diagnostic utilities rather than a clean sequential pipeline. Review file paths and avoid running file-management operations from other notebooks.

In [1]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
# Verifying your final results
from pathlib import Path
import json
import pandas as pd

base = Path("/content/drive/MyDrive/CottonGuard-AI")
results = base / "results"

files = [
    "final_results_summary.md",
    "external_evaluation_report.md",
    "resnet18_final_test_results.csv",
    "mendeley_external_metrics.json",
    "mendeley_external_confusion_matrix.csv",
]

for name in files:
    path = results / name
    print(f"\n{'=' * 60}")
    print(name)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    if path.suffix == ".md":
        print(path.read_text()[:2500])
    elif path.suffix == ".json":
        print(json.dumps(json.loads(path.read_text()), indent=2))
    elif path.suffix == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        display(df.head(10))


final_results_summary.md
Exists: False

external_evaluation_report.md
Exists: False

resnet18_final_test_results.csv
Exists: False

mendeley_external_metrics.json
Exists: False

mendeley_external_confusion_matrix.csv
Exists: False


In [3]:
# verify the original test results

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/CottonGuard-AI")
results = base / "results"

files_to_check = [
    "resnet18_final_test_results.csv",
    "resnet18_training_history.csv",
    "final_test_results.csv",
    "test_grouped.csv",
    "test.csv",
    "final_results_summary.md",
]

for name in files_to_check:
    path = results / name

    print("\n" + "=" * 60)
    print(name)

    if not path.exists():
        print("Not found")
        continue

    if path.suffix == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(5))
    else:
        print(path.read_text()[:1800])


resnet18_final_test_results.csv
Not found

resnet18_training_history.csv
Not found

final_test_results.csv
Not found

test_grouped.csv
Not found

test.csv
Not found

final_results_summary.md
Not found


In [ ]:
# RECALCULATE FINAL TEST SCORE FROM ACTUAL CHECKPOINT

import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from pathlib import Path
from PIL import Image
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from torch.utils.data import Dataset, DataLoader

# 1. Correct paths

base = Path("/content/gdrive/MyDrive/CottonGuard-AI")

test_csv = base / "results" / "data" / "test_grouped.csv"

checkpoint_path = (
    base / "models" / "deep_learning" /
    "resnet18_finetuned_best.pth"
)

# Verify files
if not test_csv.exists():
    raise FileNotFoundError(f"Test CSV not found: {test_csv}")

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {checkpoint_path}"
    )

print("Test CSV verified.")
print("Checkpoint verified.")

# 2. Class names and device

class_names = [
    "bacterial_blight",
    "curl_virus",
    "fussarium_wilt",
    "healthy"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 3. Image transformation

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 4. Load test dataset

test_df = pd.read_csv(test_csv)

# Correct any old Drive paths stored in CSV
test_df["path"] = test_df["path"].str.replace(
    "/content/drive/MyDrive/",
    "/content/gdrive/MyDrive/",
    regex=False
)

# Verify image paths
missing_images = [
    p for p in test_df["path"]
    if not Path(p).exists()
]

if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} test images are missing. "
        f"First missing path: {missing_images[0]}"
    )

print("\nTest dataset shape:", test_df.shape)
print("Test images:", len(test_df))

# 5. Create PyTorch Dataset

class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        with Image.open(row["path"]) as img:
            image = img.convert("RGB")

        image = eval_transform(image)

        label = class_names.index(row["label"])

        return image, label


test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

# 6. Load actual trained ResNet18 checkpoint

model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    len(class_names)
)

state = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(state)

model = model.to(device)
model.eval()

print("\nModel checkpoint loaded successfully.")
print("Device:", device)

# 7. Evaluate model

y_true = []
y_pred = []

with torch.inference_mode():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1).cpu().numpy()

        y_pred.extend(predictions)
        y_true.extend(labels.numpy())

# 8. Calculate metrics

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print("\n========== FINAL TEST RESULTS ==========")

print("Test images:", len(y_true))
print("Correct predictions:", sum(
    np.array(y_true) == np.array(y_pred)
))

print(f"Accuracy:         {accuracy:.6f}")
print(f"Macro Precision:  {precision:.6f}")
print(f"Macro Recall:     {recall:.6f}")
print(f"Macro F1:         {f1:.6f}")

# 9. Classification report

print("\nClassification Report:")

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

# 10. Confusion matrix

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

print("Confusion Matrix (rows=true, columns=predicted):")
print(cm)

Test CSV verified.
Checkpoint verified.

Test dataset shape: (226, 6)
Test images: 226

Model checkpoint loaded successfully.
Device: cuda

========== FINAL TEST RESULTS ==========
Test images: 226
Correct predictions: 225
Accuracy:         0.995575
Macro Precision:  0.994048
Macro Recall:     0.995690
Macro F1:         0.994814

Classification Report:
                  precision    recall  f1-score   support

bacterial_blight     1.0000    1.0000    1.0000        68
      curl_virus     1.0000    0.9828    0.9913        58
  fussarium_wilt     1.0000    1.0000    1.0000        59
         healthy     0.9762    1.0000    0.9880        41

        accuracy                         0.9956       226
       macro avg     0.9940    0.9957    0.9948       226
    weighted avg     0.9957    0.9956    0.9956       226

Confusion Matrix (rows=true, columns=predicted):
[[68  0  0  0]
 [ 0 57  0  1]
 [ 0  0 59  0]
 [ 0  0  0 41]]


In [7]:
from pathlib import Path
import re

# Correct Google Drive path
base = Path("/content/gdrive/MyDrive/CottonGuard-AI")
results = base / "results"

summary_path = results / "final_results_summary.md"

# Verify summary exists
if not summary_path.exists():
    raise FileNotFoundError(
        f"Summary file not found: {summary_path}"
    )

summary = summary_path.read_text(encoding="utf-8")

pattern = r"## 3\. Original held-out test results.*?(?=\n## 4\.)"

replacement = """## 3. Original held-out test results

The fine-tuned ResNet18 achieved on the group-aware held-out test set:

- Accuracy: 99.56%
- Correct predictions: 225 of 226 images
- Macro precision: 99.40%
- Macro recall: 99.57%
- Macro F1-score: 99.48%

These metrics were recalculated using the saved ResNet18 checkpoint and the
226 images listed in `test_grouped.csv`.

The traditional Random Forest baseline using 72 handcrafted features is
reported separately; its results should be verified against its saved
evaluation artifacts before final publication.
"""

updated, count = re.subn(
    pattern,
    replacement.rstrip(),
    summary,
    count=1,
    flags=re.DOTALL
)

if count != 1:
    print("Could not identify the exact section. No changes made.")
    print("\nCurrent section headings:")

    for line in summary.splitlines():
        if line.startswith("## "):
            print(line)

else:
    backup_path = results / "final_results_summary_backup.md"

    backup_path.write_text(summary, encoding="utf-8")
    summary_path.write_text(updated, encoding="utf-8")

    print("Updated summary:", summary_path)
    print("Backup created:", backup_path)

    print("\nUpdated section:")
    start = updated.index("## 3.")
    end = updated.index("## 4.")

    print(updated[start:end])

Updated summary: /content/gdrive/MyDrive/CottonGuard-AI/results/final_results_summary.md
Backup created: /content/gdrive/MyDrive/CottonGuard-AI/results/final_results_summary_backup.md

Updated section:
## 3. Original held-out test results

The fine-tuned ResNet18 achieved on the group-aware held-out test set:

- Accuracy: 99.56%
- Correct predictions: 225 of 226 images
- Macro precision: 99.40%
- Macro recall: 99.57%
- Macro F1-score: 99.48%

These metrics were recalculated using the saved ResNet18 checkpoint and the
226 images listed in `test_grouped.csv`.

The traditional Random Forest baseline using 72 handcrafted features is
reported separately; its results should be verified against its saved
evaluation artifacts before final publication.



In [9]:
from pathlib import Path
import pandas as pd

results = Path(
    "/content/gdrive/MyDrive/CottonGuard-AI/results"
)

print("Results directory exists:", results.exists())

print("\nFiles currently inside results:")

if results.exists():
    for item in sorted(results.rglob("*")):
        if item.is_file():
            print(item.relative_to(results))

for filename in [
    "traditional_ml_validation_results.csv",
    "final_test_results.csv",
    "final_results_summary.md"
]:
    path = results / filename

    print(f"\n{'=' * 60}")
    print(filename)

    if path.exists() and path.suffix == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(15))

    elif path.exists():
        print(path.read_text(encoding="utf-8")[:5000])

    else:
        print("File not found")

Results directory exists: True

Files currently inside results:
audit_review/corrupted_images.csv
audit_review/cross_class_duplicates.json
audit_review/dataset_audit_summary.csv
audit_review/deduplicated_image_index.csv
audit_review/image_metadata.csv
audit_review/near_duplicate_mixed_label_groups.csv
audit_review/test_grouped_candidate.csv
audit_review/train_grouped_candidate.csv
audit_review/validation_grouped_candidate.csv
background_sensitivity_20_images.csv
data/deduplicated_image_index.csv
data/near_duplicate_candidates.csv
data/perceptual_grouped_image_index.csv
data/test.csv
data/test_grouped.csv
data/train.csv
data/train_grouped.csv
data/validation.csv
data/validation_grouped.csv
external_evaluation_preliminary.csv
final_results_summary.md
final_results_summary_backup.md
final_test_results.csv
grouped_validation_results.csv
mendeley_external_confusion_matrix.csv
mendeley_external_metrics.json
mendeley_external_predictions.csv
metrics/final_test_results.csv
metrics/grouped_vali

,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,Training_and_prediction_seconds
0,Random Forest,0.974895,0.972662,0.974001,0.973156,2.04
1,SVM,0.937238,0.943478,0.935795,0.938814,0.40
2,Decision Tree,0.903766,0.904600,0.908870,0.906043,0.08
3,KNN,0.874477,0.876362,0.869945,0.870446,0.08



final_test_results.csv
Shape: (1, 4)
Columns: ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']


,Accuracy,Macro Precision,Macro Recall,Macro F1
0,0.964602,0.960465,0.956654,0.95832



final_results_summary.md
# CottonGuard AI — Final Results Summary

**Report date:** 2026-11-01

## 1. Project overview

CottonGuard AI is a cotton leaf disease classification project using
a fine-tuned ResNet18 deep learning model and traditional machine-learning
baselines.

The four classes are:
- Bacterial blight
- Curl virus
- Fusarium wilt
- Healthy

## 2. Dataset preparation

- Original dataset images: 1,716
- Unique image contents after exact duplicate auditing: 1,591
- Training images: 1,126
- Validation images: 239
- Original held-out test images: 226
- External Mendeley images evaluated: 958

The original train, validation, and test split was group-aware to reduce
image leakage between splits.

## 3. Original held-out test results

The fine-tuned ResNet18 achieved on the group-aware held-out test set:

- Accuracy: 99.56%
- Correct predictions: 225 of 226 images
- Macro precision: 99.40%
- Macro recall: 99.57%
- Macro F1-score: 99.48%

These metrics were recalculated using the

In [11]:
# identify the source of the traditional ML test metrics
from pathlib import Path
import pandas as pd

base = Path("/content/gdrive/MyDrive/CottonGuard-AI")

print("MODEL FILES")
for path in sorted((base / "models").glob("*")):
    print(path.name)

print("\nRESULT FILES")
for path in sorted((base / "results").glob("*.csv")):
    print(f"\n--- {path.name} ---")
    df = pd.read_csv(path)
    print("Columns:", df.columns.tolist())
    print("Rows:", len(df))
    if len(df) <= 10:
        display(df)

MODEL FILES
decision_tree_grouped.joblib
deep_learning
knn_grouped.joblib
random_forest_grouped.joblib
svm_grouped.joblib
traditional_ml

RESULT FILES

--- background_sensitivity_20_images.csv ---
Columns: ['path', 'true_label', 'condition', 'prediction', 'correct', 'true_class_probability', 'predicted_class_probability', 'prob_bacterial_blight', 'prob_curl_virus', 'prob_fussarium_wilt', 'prob_healthy']
Rows: 60

--- external_evaluation_preliminary.csv ---
Columns: ['image', 'image_path', 'external_folder', 'true_label', 'predicted_label', 'confidence', 'correct']
Rows: 17

--- final_test_results.csv ---
Columns: ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
Rows: 1


,Accuracy,Macro Precision,Macro Recall,Macro F1
0,0.964602,0.960465,0.956654,0.95832



--- grouped_validation_results.csv ---
Columns: ['Model', 'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Training Time (seconds)']
Rows: 4


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Training Time (seconds)
0,Random Forest,0.983264,0.985523,0.982456,0.983555,1.25
1,SVM,0.928870,0.935344,0.929301,0.930806,0.35
2,Decision Tree,0.891213,0.900011,0.885588,0.890683,0.09
3,KNN,0.870293,0.874841,0.868660,0.869653,0.01



--- mendeley_external_confusion_matrix.csv ---
Columns: ['Unnamed: 0', 'bacterial_blight', 'curl_virus', 'fussarium_wilt', 'healthy']
Rows: 4


,Unnamed: 0,bacterial_blight,curl_virus,fussarium_wilt,healthy
0,bacterial_blight,82,2,0,35
1,curl_virus,0,0,0,14
2,fussarium_wilt,4,2,0,30
3,healthy,20,6,2,761



--- mendeley_external_predictions.csv ---
Columns: ['image', 'relative_path', 'true_label', 'predicted_label', 'confidence', 'correct']
Rows: 958

--- near_duplicate_candidates.csv ---
Columns: ['path_a', 'label_a', 'split_a', 'path_b', 'label_b', 'split_b', 'hash_distance']
Rows: 77

--- original_vs_cropped_predictions.csv ---
Columns: ['path', 'true_label', 'original_prediction', 'cropped_prediction', 'original_correct', 'cropped_correct', 'prediction_changed', 'original_confidence', 'cropped_confidence']
Rows: 226

--- traditional_ml_validation_results.csv ---
Columns: ['Model', 'Accuracy', 'Precision_macro', 'Recall_macro', 'F1_macro', 'Training_and_prediction_seconds']
Rows: 4


,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,Training_and_prediction_seconds
0,Random Forest,0.974895,0.972662,0.974001,0.973156,2.04
1,SVM,0.937238,0.943478,0.935795,0.938814,0.40
2,Decision Tree,0.903766,0.904600,0.908870,0.906043,0.08
3,KNN,0.874477,0.876362,0.869945,0.870446,0.08


In [13]:
# evaluate the grouped traditional ML models

from pathlib import Path

base = Path("/content/gdrive/MyDrive/CottonGuard-AI")

print("Notebook and Python files:")
for pattern in ["*.ipynb", "*.py"]:
    for path in sorted(base.rglob(pattern)):
        if "external_dataset" not in str(path):
            print(path.relative_to(base))

print("\nProject folders:")
for path in sorted(base.iterdir()):
    print("[DIR]" if path.is_dir() else "[FILE]", path.name)

Notebook and Python files:
notebooks/01_dataset_audit.ipynb
notebooks/02_traditional_ml.ipynb
notebooks/03_resnet18_training.ipynb
notebooks/04_model_explainability.ipynb
notebooks/05_external_evaluation.ipynb
notebooks/06_results_verification.ipynb
notebooks/CottonGuard_AI_Cleaned_Workflow.ipynb
app/streamlit_app.py
app.py
src/__init__.py
src/config.py
src/data/__init__.py
src/data/prepare_splits.py
src/evaluation/__init__.py
src/evaluation/evaluate_external.py
src/evaluation/evaluate_resnet.py
src/evaluation/evaluate_traditional_ml.py
src/evaluation/gradcam.py
src/features/__init__.py
src/features/image_features.py
src/predict.py
src/train_resnet.py
src/train_traditional_ml.py
tests/test_predict.py

Project folders:
[FILE] README.md
[FILE] REFACTOR_NOTES.md
[DIR] app
[FILE] app.py
[DIR] data
[DIR] external_dataset
[DIR] models
[DIR] notebooks
[FILE] requirements.txt
[DIR] results
[DIR] src
[DIR] tests


In [15]:
from pathlib import Path

drive = Path("/content/gdrive/MyDrive")

extensions = {".ipynb", ".py"}
keywords = ("cotton", "machine", "learning", "feature", "model", "train")

print("Searching Google Drive for relevant notebooks and scripts...\n")

found = []

for path in drive.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    # Skip large datasets and dependency folders
    parts = {part.lower() for part in path.parts}
    if parts.intersection({
        ".git", "node_modules", "__pycache__",
        "site-packages", ".venv"
    }):
        continue

    if any(word in path.name.lower() for word in keywords):
        found.append(path)

for path in found[:100]:
    print(path)

print(f"\nTotal relevant files found: {len(found)}")

Searching Google Drive for relevant notebooks and scripts...



KeyboardInterrupt: 